In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
silver_orders = spark.table(f"{catalogo}.{esquema_source}.silver_orders")
silver_order_products = spark.table(f"{catalogo}.{esquema_source}.silver_order_products")
silver_products = spark.table(f"{catalogo}.{esquema_source}.silver_products")
silver_orders_products_detail = spark.table(f"{catalogo}.{esquema_source}.silver_orders_products_detail")

## Ingesta tablas Golden

In [0]:
# Productos más vendidos
gold_top_products = (
    silver_orders_products_detail
    .groupBy("product_id", "product_name", "department", "aisle")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("reordered") == 1, 1).otherwise(0)).alias("total_reorders")
    )
    .orderBy(F.col("total_orders").desc())
)
gold_top_products.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.gold_top_products")

In [0]:
# Tasa de reorder por departamento
gold_reorder_rate_department = (
    silver_orders_products_detail
    .groupBy("department")
    .agg(
        F.count("*").alias("total_items"),
        F.sum(F.when(F.col("reordered") == 1, 1).otherwise(0)).alias("total_reorders"),
        F.round(
            F.sum(F.when(F.col("reordered") == 1, 1).otherwise(0)) * 100.0 / F.count("*"), 2
        ).alias("reorder_rate_pct")
    )
    .orderBy(F.col("reorder_rate_pct").desc())
)
gold_reorder_rate_department.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.gold_reorder_rate_department")

In [0]:
# Hábitos de compra por día y franja horaria
gold_purchase_patterns = (
    silver_orders_products_detail
    .groupBy("order_day_name", "order_time_category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items")
    )
    .orderBy(F.col("total_orders").desc())
)
gold_purchase_patterns.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.gold_purchase_patterns")

In [0]:
# Segmentación de clientes por frecuencia
gold_user_segments = (
    silver_orders_products_detail
    .groupBy("user_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.avg("days_since_prior_order").alias("avg_days_between_orders")
    )
    .withColumn(
        "user_segment",
        F.when(F.col("total_orders") >= 50, "High frequency")
         .when(F.col("total_orders").between(20, 49), "Medium frequency")
         .otherwise("Low frequency")
    )
)
gold_user_segments.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.gold_user_segments")